# Unittest Basics

> Generated from [`09-Unittest-Basics.py`](09-Unittest-Basics.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
Python's built-in `unittest` testing framework — writing automated checks (test cases, assertions, setup/teardown, mocking) that catch regressions instead of manually re-verifying behavior.

### Key points
- **`class TestX(unittest.TestCase)`** — subclass `TestCase`; every method starting with `test_` is a separate check.
- **Common assertions** — `assertEqual`, `assertTrue`/`assertFalse`, `assertIsNone`, `assertIn`, `assertAlmostEqual` (for floats), `assertRaises` (as context manager, to check an exception is raised).
- **`setUp()` / `tearDown()`** — run before/after *every* test method in the class, for fresh state and cleanup.
- **Running tests** — `unittest.TestLoader().loadTestsFromModule(...)` + `TextTestRunner().run(suite)`; check `result.testsRun`, `.failures`, `.errors`.
- **`pytest`** (third-party, not in this file's imports but referenced) is the more popular modern alternative — simpler syntax, no class boilerplate — but `unittest` ships with Python, nothing to install.
- **`unittest.mock`** — `Mock()` stands in for any object, records calls; `patch("module.thing", return_value=...)` temporarily replaces a real object during a `with` block, restoring it afterward.
- **`assertRaises(...) as ctx`** — inspect the actual exception via `ctx.exception` (e.g. check its message).
- **`subTest(a=a, b=b)`** — parameterize a test loop so each iteration is reported as a separate result, instead of stopping at the first failure.
- **`@unittest.skip("reason")`** / **`@unittest.skipIf(condition, "reason")`** — skip tests conditionally.
- **Test discovery** — `python -m unittest discover -s tests -p 'test_*.py'` auto-finds tests across a project without importing each file by hand.

### Gotchas (the part worth revising twice)
- `patch()` only replaces the target for the duration of its `with` block (or decorated function) — the real object is restored automatically afterward; don't forget it's scoped.
- Without `subTest()`, a loop of assertions stops reporting at the first failure — `subTest()` lets every case in the loop report independently.
- `setUp()`/`tearDown()` run before/after **each** test method, not once for the whole class — don't rely on state persisting between test methods.
- A skipped test (`@unittest.skip`) still counts in the run summary as "skipped", separate from pass/fail — check `result.skipped` if you need that count.

### When it matters
Any code you want to protect from silent regressions — refactoring safely, verifying edge cases (like divide-by-zero), or replacing slow/external dependencies (network calls) with mocks during tests.

### One-line recap example
```python
import unittest
class TestAdd(unittest.TestCase):
    def test_add(self):
        self.assertEqual(1 + 1, 2)
```

In [ ]:
import unittest

## 01. unittest - Introduction

- Python's built-in testing framework - write automated checks that catch
  regressions instead of manually re-checking behavior by hand.
- pytest (third-party, `pip install pytest`) is the more popular modern
  choice - simpler syntax, no boilerplate class needed - but unittest
  ships with Python and needs nothing installed.

## 02. Code Under Test

In [ ]:
def add(a, b):
    return a + b

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

## 03. Writing a Test Case

- Subclass unittest.TestCase; each method starting with `test_` is a check.

In [ ]:
class TestMathFunctions(unittest.TestCase):

    def test_add_positive_numbers(self):
        self.assertEqual(add(2, 3), 5)

    def test_add_negative_numbers(self):
        self.assertEqual(add(-1, -1), -2)

    def test_divide_normal(self):
        self.assertEqual(divide(10, 2), 5)

    def test_divide_by_zero_raises(self):
        with self.assertRaises(ValueError):
            divide(10, 0)

    def test_common_assertions(self):
        self.assertTrue(3 > 2)
        self.assertFalse(2 > 3)
        self.assertIsNone(None)
        self.assertIn(3, [1, 2, 3])
        self.assertAlmostEqual(0.1 + 0.2, 0.3, places=5)

## 04. setUp() and tearDown() - Run Before/After Each Test

In [ ]:
class TestWithSetup(unittest.TestCase):
    def setUp(self):
        self.data = [1, 2, 3]  # fresh state before every test method

    def tearDown(self):
        self.data = None  # cleanup after every test method

    def test_data_length(self):
        self.assertEqual(len(self.data), 3)

    def test_data_sum(self):
        self.assertEqual(sum(self.data), 6)

print("\n# 02-04. Test Cases Defined (see run below)")

## 05. Running the Tests

In [ ]:
suite = unittest.TestLoader().loadTestsFromModule(__import__("__main__"))
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

## 06. Reading Results

In [ ]:
print(f"tests run: {result.testsRun}, failures: {len(result.failures)}, errors: {len(result.errors)}")

## 07. pytest Equivalent (reference, not executed - requires `pip install pytest`)

## 08. unittest.mock - Mock() and patch() for Replacing Dependencies

- Mock() stands in for any object; calls/attribute access are recorded.
- patch() temporarily replaces a real object (e.g. an external API call)
  with a Mock, then restores the original when the `with` block ends.

In [ ]:
from unittest.mock import Mock, patch

def fetch_price(client, symbol):
    return client.get_price(symbol)  # imagine this hits a real network API

print("\n# 08. unittest.mock - Mock() and patch()")
fake_client = Mock()
fake_client.get_price.return_value = 42.50
print("mocked call result:", fetch_price(fake_client, "AAPL"))
print("was it called with 'AAPL'?", fake_client.get_price.call_args)

def get_current_dir():
    import os
    return os.getcwd()

with patch("os.getcwd", return_value="/fake/path"):
    print("patched os.getcwd():", get_current_dir())
print("real os.getcwd() after the `with` block ends:", get_current_dir())

## 09. assertRaises as a Context Manager - Inspecting the Exception

In [ ]:
try:
    class TestDivideMessage(unittest.TestCase):
        def test_message(self):
            with self.assertRaises(ValueError) as ctx:
                divide(10, 0)
            self.assertEqual(str(ctx.exception), "Cannot divide by zero")

    single_result = unittest.TextTestRunner(verbosity=0).run(
        unittest.TestLoader().loadTestsFromTestCase(TestDivideMessage)
    )
    print("passed:", single_result.wasSuccessful())
except Exception as e:
    print("unexpected error:", e)

## 10. subTest() - Parameterizing Tests Without Extra Test Methods

In [ ]:
class TestAddParameterized(unittest.TestCase):
    def test_add_many_cases(self):
        cases = [(1, 1, 2), (2, 3, 5), (-1, 1, 0), (0, 0, 0)]
        for a, b, expected in cases:
            with self.subTest(a=a, b=b):  # each iteration reported separately
                self.assertEqual(add(a, b), expected)

print("\n# 10. subTest() Parameterized Tests")
sub_result = unittest.TextTestRunner(verbosity=0).run(
    unittest.TestLoader().loadTestsFromTestCase(TestAddParameterized)
)
print("all cases passed:", sub_result.wasSuccessful())

## 11. Skipping Tests

In [ ]:
class TestSkipping(unittest.TestCase):
    @unittest.skip("demonstration - always skipped")
    def test_always_skipped(self):
        self.fail("should never run")

    @unittest.skipIf(1 + 1 != 2, "math is broken, skip")
    def test_runs_because_condition_false(self):
        self.assertEqual(1 + 1, 2)

print("\n# 11. Skipping Tests")
skip_result = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestLoader().loadTestsFromTestCase(TestSkipping)
)
print("skipped count:", len(skip_result.skipped))

## 12. Test Discovery (reference - not run here)